# Sewage Defect — Tier 2 Segmentation Training on SageMaker (GPU)

Train **yolo26m-seg** on the SAM-generated pseudo-label dataset. Auto-locates the
`sewage-yolo26-seg/` folder wherever you uploaded it.

**Prerequisite** — run `src/generate_seg_labels.py` locally first to produce
`sewage-yolo26-seg/{train,valid,test}/{images,labels}/` (polygon labels), then
upload that folder to SageMaker (file browser).

**Outputs** — stages `best.pt`, ONNX (fp16), `classes.json`, `metadata.yaml`,
per-class mAP CSV, and qualitative good/missed/false samples into a single
`model/` folder next to this notebook. Right-click → Download in the file browser.
HuggingFace upload is a separate notebook (`sagemaker_hf_upload.ipynb`).

## 1. Environment check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import platform, torch
print()
print('python  :', platform.python_version())
print('torch   :', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device  :', torch.cuda.get_device_name(0))
    print('VRAM    :', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GiB')

## 2. Install dependencies

In [ ]:
# IMPORTANT: pin torchvision to match the pre-installed torch (2.8) — the
# SageMaker image ships torchvision==0.24 which is incompatible with torch
# 2.8, and that mismatch crashes dataloader workers with cryptic
# `_pickle.PicklingError: ... numpy._core.multiarray` errors mid-epoch.
#
# After running this cell, restart the kernel (Kernel → Restart) so the new
# torchvision is actually loaded, then re-run from Cell 3 onwards.
!pip install -q -U \
    'ultralytics>=8.4.51' \
    'pycocotools>=2.0.11' \
    'onnx>=1.17' 'onnxslim>=0.1.71' 'onnxruntime' \
    matplotlib opencv-python-headless

!pip install -q 'torchvision==0.23.*'

# Sanity-check after install (will only be accurate after kernel restart).
import torch, torchvision
print(f'torch={torch.__version__}  torchvision={torchvision.__version__}')
print(f'compatible: {torch.__version__.startswith("2.8") and torchvision.__version__.startswith("0.23")}')

## 3. Locate the seg dataset

Auto-detects `sewage-yolo26-seg/` regardless of where SageMaker placed the notebook.
Add to `CANDIDATES` if you put it elsewhere.

In [ ]:
from pathlib import Path

CANDIDATES = [
    Path('sewage-yolo26-seg'),
    Path('ass3/sewage-yolo26-seg'),
    Path('../sewage-yolo26-seg'),
    Path('../ass3/sewage-yolo26-seg'),
    Path.cwd() / 'sewage-yolo26-seg',
]
DATASET = next(
    (p for p in CANDIDATES
     if (p / 'train' / 'images').exists() and (p / 'valid' / 'images').exists()),
    None,
)
assert DATASET is not None, (
    f'sewage-yolo26-seg not found. cwd={Path.cwd()}. tried: '
    + ', '.join(str(p) for p in CANDIDATES)
)
DATASET = DATASET.resolve()
print('dataset:', DATASET)
!ls {DATASET}

## 3.1 Copy images from the bbox dataset

Ultralytics resolves the image directory before applying `/images/` → `/labels/`
substitution to find labels. That means a **directory-level symlink** for
`sewage-yolo26-seg/train/images` → `sewage-yolo26/train/images` makes Ultralytics
look up labels at `sewage-yolo26/train/labels/` (bbox labels, 5 fields) instead
of `sewage-yolo26-seg/train/labels/` (polygon labels) — silent disaster.

So we copy the images instead. 980 files × ~50KB ≈ 49MB extra disk. Worth it
for a working pipeline.

If a previous run created a directory symlink at `images`, this cell tears it
down and rebuilds with real copies. Idempotent — already-copied files are skipped.

In [ ]:
import shutil

BBOX_CANDIDATES = [
    DATASET.parent / 'sewage-yolo26',
    Path('sewage-yolo26'),
    Path('ass3/sewage-yolo26'),
    Path('../sewage-yolo26'),
    Path('../ass3/sewage-yolo26'),
    Path.cwd() / 'sewage-yolo26',
]
BBOX = next(
    (p for p in BBOX_CANDIDATES
     if all((p / s / 'images').exists() for s in ('train', 'valid', 'test'))),
    None,
)
assert BBOX is not None, (
    'bbox dataset (sewage-yolo26) not found — upload it to SageMaker next to '
    'sewage-yolo26-seg so we can copy its images. '
    f'tried: {[str(p) for p in BBOX_CANDIDATES]}'
)
BBOX = BBOX.resolve()
print('bbox dataset (image source):', BBOX)
print()

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

for split in ('train', 'valid', 'test'):
    src_dir = BBOX / split / 'images'
    dst_dir = DATASET / split / 'images'
    assert src_dir.is_dir(), f'missing source dir: {src_dir}'

    # Tear down any prior directory-symlink (left behind by older versions of this cell).
    if dst_dir.is_symlink():
        dst_dir.unlink()
    dst_dir.mkdir(parents=True, exist_ok=True)

    copied = skipped = removed_broken = 0
    src_names = {p.name for p in src_dir.iterdir() if p.suffix.lower() in IMG_EXTS}

    # Remove any stale file-symlinks in dst that are broken (point to local Mac paths).
    for p in list(dst_dir.iterdir()):
        if p.is_symlink() and not p.exists():
            p.unlink()
            removed_broken += 1

    # Copy missing files (or replace lingering symlinks with real copies).
    for name in src_names:
        src = src_dir / name
        dst = dst_dir / name
        if dst.is_symlink():
            dst.unlink()
        if dst.is_file():
            skipped += 1
            continue
        shutil.copy2(src, dst)
        copied += 1

    print(f'[{split}] copied={copied} already_present={skipped} removed_broken={removed_broken}')

print()
print('Final counts:')
print(f"{'split':<6} {'images':>7} {'labels':>7}")
for split in ('train', 'valid', 'test'):
    img_dir = DATASET / split / 'images'
    lbl_dir = DATASET / split / 'labels'
    n_img = sum(1 for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS)
    n_lbl = sum(1 for _ in lbl_dir.glob('*.txt'))
    print(f'{split:<6} {n_img:>7} {n_lbl:>7}')

## 5. Train yolo26m-seg

- Base: `yolo26m-seg.pt` (COCO-segmentation pretrained — ultralytics auto-downloads).
- imgsz=640, 200 epochs, MuSGD, cosine LR, patience=50.
- Heavy aug (copy_paste 0.3, mixup 0.1) compensates for the small (686-img) train split
  and naturally over-samples minority classes from copy-paste donors.
- No per-class loss weighting — Ultralytics does not expose `cls_weights` as a
  training kwarg.
- `save_period=10` writes a checkpoint every 10 epochs in addition to the standard
  `last.pt` (latest) and `best.pt` (best val mAP) that Ultralytics always keeps.

**Resume**:
- If the kernel dies or the instance reboots mid-training, set `RESUME = True`
  and re-run this cell. Ultralytics picks up `runs/segment/tier2_seg_sagemaker/weights/last.pt`
  and continues from the epoch it crashed at. All hyperparameters are restored
  from the saved checkpoint — the `train_kwargs` you set here is ignored on resume.
- If you want to start fresh, leave `RESUME = False` (default).

**Smoke test**:
- Set `SMOKE_TEST = True` for a 3-epoch dry run (~5 min) before the full run.

In [ ]:
SMOKE_TEST = False   # True → 3-epoch dry run (~5 min)
RESUME     = False   # True → continue from out/seg/weights/last.pt

import yaml, torch
from pathlib import Path
from ultralytics import YOLO

# --- Self-contained setup: works regardless of which earlier cells were run. ---

# 1. Locate the seg dataset if not already in scope.
if 'DATASET' not in globals():
    for cand in (Path('sewage-yolo26-seg'),
                 Path('ass3/sewage-yolo26-seg'),
                 Path('../sewage-yolo26-seg'),
                 Path('../ass3/sewage-yolo26-seg'),
                 Path.cwd() / 'sewage-yolo26-seg'):
        if (cand / 'train' / 'images').exists() and (cand / 'valid' / 'images').exists():
            DATASET = cand.resolve()
            print('auto-located DATASET:', DATASET)
            break
    else:
        raise FileNotFoundError(
            f'sewage-yolo26-seg not found. cwd={Path.cwd()}. '
            'Run Cell 3 (locate dataset) first.'
        )

# 2. Re-normalise data.yaml (absolute path + task=segment) every time.
data_yaml = DATASET / 'data.yaml'
assert data_yaml.exists(), f'missing {data_yaml} — re-upload the seg dataset or check the path.'
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)
cfg['path']  = str(DATASET)
cfg['train'] = 'train/images'
cfg['val']   = 'valid/images'
cfg['test']  = 'test/images'
cfg['task']  = 'segment'
with open(data_yaml, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

names = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][i] for i in sorted(cfg['names'])]
print('classes:', names)

# --- Training ---

LAST_CKPT = Path('out/seg/weights/last.pt')

if RESUME:
    assert LAST_CKPT.exists(), (
        f'RESUME=True but no checkpoint at {LAST_CKPT}. '
        'Set RESUME=False to start fresh, or fix the path.'
    )
    print(f'Resuming from {LAST_CKPT}')
    model = YOLO(LAST_CKPT)
    results = model.train(resume=True)
else:
    model = YOLO('yolo26m-seg.pt')

    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    BATCH = 12 if vram_gb >= 20 else 6
    print(f'using batch={BATCH} for {vram_gb:.0f} GiB GPU')

    train_kwargs = dict(
        data=str(data_yaml),
        task='segment',
        epochs=3 if SMOKE_TEST else 200,
        imgsz=640,
        batch=BATCH,
        device=0,
        optimizer='MuSGD',
        cos_lr=True,
        patience=50,
        plots=True,
        project='out',        # short output root → out/seg/...
        name='seg',
        exist_ok=True,
        save_period=10,       # checkpoint every 10 epochs (extra granularity for resume)
        mosaic=1.0, close_mosaic=20,
        mixup=0.10, copy_paste=0.30,
        scale=0.6, degrees=10.0, translate=0.10,
        hsv_v=0.30, hsv_s=0.60, fliplr=0.5,
        amp=True,
        workers=4,            # set to 0 if PicklingError persists
    )
    results = model.train(**train_kwargs)

# Capture the real save_dir from Ultralytics — never hardcode it.
TRAIN_DIR = Path(model.trainer.save_dir)
BEST = TRAIN_DIR / 'weights' / 'best.pt'
print()
print('train save_dir:', TRAIN_DIR)
print('best weights  :', BEST)
print('last weights  :', TRAIN_DIR / 'weights' / 'last.pt')
assert BEST.exists(), f'best.pt not found at {BEST}'

## 5. Train yolo26m-seg

- Base: `yolo26m-seg.pt` (COCO-segmentation pretrained — ultralytics auto-downloads).
- imgsz=640, 200 epochs, MuSGD, cosine LR, patience=50.
- Heavy aug (copy_paste 0.3, mixup 0.1) compensates for the small (686-img) train split
  and naturally over-samples minority classes from copy-paste donors.
- No per-class loss weighting — Ultralytics does not expose `cls_weights` as a
  training kwarg. If you really need it, the way is to patch the loss function
  via a callback (`trainer.add_callback('on_train_start', ...)`). Skipping here.

Set `SMOKE_TEST = True` first for a 3-epoch dry run (~5 min) before the full run.

In [ ]:
best = YOLO(BEST)
test_metrics = best.val(
    data=str(data_yaml), split='test', imgsz=640, device=0,
    conf=0.001, iou=0.6, augment=True, plots=True,
    project='out', name='seg_test', exist_ok=True,
)
TEST_DIR = Path(test_metrics.save_dir)
print()
print('test save_dir:', TEST_DIR)
print('TEST  | Box  mAP@0.5  ', f'{test_metrics.box.map50:.4f}')
print('TEST  | Box  mAP@.5:.95', f'{test_metrics.box.map:.4f}')
print('TEST  | Mask mAP@0.5  ', f'{test_metrics.seg.map50:.4f}')
print('TEST  | Mask mAP@.5:.95', f'{test_metrics.seg.map:.4f}')

## 6. Evaluate on the test split — overall mAP

In [ ]:
import csv

ap_idx = list(test_metrics.box.ap_class_index)
results_by_cid = {}
for i, cid in enumerate(ap_idx):
    bp, br, bap50, bap = test_metrics.box.class_result(i)
    sp, sr, sap50, sap = test_metrics.seg.class_result(i)
    results_by_cid[int(cid)] = (bp, br, bap50, bap, sp, sr, sap50, sap)

header = (
    f"{'class':<20} {'P_box':>6} {'R_box':>6} {'AP50_box':>9} {'AP_box':>7}"
    f"  | {'P_seg':>6} {'R_seg':>6} {'AP50_seg':>9} {'AP_seg':>7}"
)
print(header)
print('-' * len(header))

rows = []
for cid, name in enumerate(names):
    if cid in results_by_cid:
        bp, br, bap50, bap, sp, sr, sap50, sap = results_by_cid[cid]
    else:
        bp = br = bap50 = bap = sp = sr = sap50 = sap = 0.0
    rows.append([name, bp, br, bap50, bap, sp, sr, sap50, sap])
    print(f'{name:<20} {bp:>6.3f} {br:>6.3f} {bap50:>9.3f} {bap:>7.3f}'
          f'  | {sp:>6.3f} {sr:>6.3f} {sap50:>9.3f} {sap:>7.3f}')

PER_CLASS_CSV = TEST_DIR / 'per_class_metrics.csv'
with open(PER_CLASS_CSV, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['class', 'P_box', 'R_box', 'AP50_box', 'AP_box',
                'P_seg', 'R_seg', 'AP50_seg', 'AP_seg'])
    w.writerows(rows)
print()
print('Saved CSV:', PER_CLASS_CSV.resolve())

## 6.1 Per-class metrics (Box + Mask)

Prints a per-class precision / recall / AP@0.5 / AP@0.5:0.95 table for both
bounding boxes and instance masks, then saves to CSV.

In [ ]:
import cv2, numpy as np, shutil

N_PER_BIN = 4
CONF_VIS = 0.25
QUAL_DIR = TEST_DIR / 'qualitative'
if QUAL_DIR.exists():
    shutil.rmtree(QUAL_DIR)
QUAL_DIR.mkdir(parents=True, exist_ok=True)

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
test_imgs = sorted(
    p for p in (DATASET / 'test' / 'images').iterdir()
    if p.suffix.lower() in IMG_EXTS
)
print(f'predicting on {len(test_imgs)} test images...')
predictions = best.predict(
    source=[str(p) for p in test_imgs],
    conf=CONF_VIS, iou=0.45, imgsz=640, device=0,
    save=False, verbose=False,
)

def color_for(cid):
    rs = np.random.RandomState(cid * 37 + 11)
    return tuple(int(c) for c in rs.randint(64, 240, 3))

def draw_gt(img_bgr, lbl_path):
    h, w = img_bgr.shape[:2]
    out = img_bgr.copy()
    if not lbl_path.exists():
        return out
    overlay = out.copy()
    for line in lbl_path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        cls = int(parts[0])
        coords = list(map(float, parts[1:]))
        if len(coords) < 6:
            continue
        pts = np.array(
            [(coords[i] * w, coords[i + 1] * h) for i in range(0, len(coords) - 1, 2)],
            dtype=np.int32,
        )
        col = color_for(cls)
        cv2.fillPoly(overlay, [pts], col)
        cv2.polylines(out, [pts], True, col, 2)
        cv2.putText(out, names[cls], tuple(pts[0]), cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, col, 2, cv2.LINE_AA)
    cv2.addWeighted(overlay, 0.35, out, 0.65, 0, dst=out)
    return out

def banner(img, text):
    out = img.copy()
    cv2.rectangle(out, (0, 0), (out.shape[1], 36), (0, 0, 0), -1)
    cv2.putText(out, text, (10, 26), cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (255, 255, 255), 2, cv2.LINE_AA)
    return out

def side_by_side(gt_img, pred_img, title):
    th = 480
    g = cv2.resize(gt_img, (int(gt_img.shape[1] * th / gt_img.shape[0]), th))
    p = cv2.resize(pred_img, (int(pred_img.shape[1] * th / pred_img.shape[0]), th))
    g = banner(g, 'GT')
    p = banner(p, 'Pred')
    sep = np.full((th, 4, 3), 255, dtype=np.uint8)
    side = np.hstack([g, sep, p])
    return banner(side, title)

bins = {cid: {'good': [], 'missed': [], 'false': []} for cid in range(len(names))}
for img_path, pred in zip(test_imgs, predictions):
    lbl = DATASET / 'test' / 'labels' / (img_path.stem + '.txt')
    gt_classes = set()
    if lbl.exists():
        for line in lbl.read_text().splitlines():
            line = line.strip()
            if line:
                gt_classes.add(int(line.split()[0]))

    if pred.boxes is None or len(pred.boxes) == 0:
        pred_max_conf = {}
    else:
        cls_arr = pred.boxes.cls.cpu().numpy().astype(int)
        conf_arr = pred.boxes.conf.cpu().numpy()
        pred_max_conf = {}
        for c, f in zip(cls_arr, conf_arr):
            pred_max_conf[int(c)] = max(pred_max_conf.get(int(c), 0.0), float(f))
    pred_classes = set(pred_max_conf)

    for cid in range(len(names)):
        in_gt = cid in gt_classes
        in_pred = cid in pred_classes
        score = pred_max_conf.get(cid, 0.0)
        if in_gt and in_pred:
            bins[cid]['good'].append((score, img_path, pred))
        elif in_gt and not in_pred:
            bins[cid]['missed'].append((0.0, img_path, pred))
        elif not in_gt and in_pred:
            bins[cid]['false'].append((score, img_path, pred))

summary = []
for cid, name in enumerate(names):
    safe = name.replace(' ', '_').replace('/', '_')
    counts = {}
    for kind in ('good', 'missed', 'false'):
        items = bins[cid][kind]
        items.sort(key=lambda x: x[0], reverse=True)
        counts[kind] = len(items)
        for rank, (score, img_path, pred) in enumerate(items[:N_PER_BIN], 1):
            img = cv2.imread(str(img_path))
            gt_vis = draw_gt(img, DATASET / 'test' / 'labels' / (img_path.stem + '.txt'))
            pred_vis = pred.plot()
            title = f'[{kind.upper()}] cls={name} conf={score:.2f} file={img_path.stem}'
            out = side_by_side(gt_vis, pred_vis, title)
            cv2.imwrite(str(QUAL_DIR / f'{safe}__{kind}__{rank:02d}__{img_path.stem}.jpg'), out)
    summary.append((name, counts['good'], counts['missed'], counts['false']))

print()
print(f"{'class':<20} {'good':>5} {'missed':>7} {'false':>6}")
print('-' * 42)
for name, g, m, f in summary:
    print(f'{name:<20} {g:>5} {m:>7} {f:>6}')
print()
print('Saved qualitative samples to', QUAL_DIR.resolve())

## 6.2 Qualitative samples — good / missed / false per class

For each class, saves up to `N_PER_BIN` side-by-side `GT (left) | Pred (right)`
JPEGs into `runs/segment/qualitative/`:

- `good`    — class present in both GT and predictions
- `missed`  — class in GT but not predicted (false negative)
- `false`   — class predicted but not in GT (false positive)

Sorted by predicted confidence so the top examples are the most representative.

In [ ]:
import cv2, numpy as np, shutil
from pathlib import Path

N_PER_BIN = 4
CONF_VIS = 0.25
QUAL_DIR = Path('runs/segment/qualitative')
if QUAL_DIR.exists():
    shutil.rmtree(QUAL_DIR)
QUAL_DIR.mkdir(parents=True, exist_ok=True)

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
test_imgs = sorted(
    p for p in (DATASET / 'test' / 'images').iterdir()
    if p.suffix.lower() in IMG_EXTS
)
print(f'predicting on {len(test_imgs)} test images...')
predictions = best.predict(
    source=[str(p) for p in test_imgs],
    conf=CONF_VIS, iou=0.45, imgsz=640, device=0,
    save=False, verbose=False,
)

def color_for(cid):
    rs = np.random.RandomState(cid * 37 + 11)
    return tuple(int(c) for c in rs.randint(64, 240, 3))

def draw_gt(img_bgr, lbl_path):
    h, w = img_bgr.shape[:2]
    out = img_bgr.copy()
    if not lbl_path.exists():
        return out
    overlay = out.copy()
    for line in lbl_path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        cls = int(parts[0])
        coords = list(map(float, parts[1:]))
        if len(coords) < 6:
            continue
        pts = np.array(
            [(coords[i] * w, coords[i + 1] * h) for i in range(0, len(coords) - 1, 2)],
            dtype=np.int32,
        )
        col = color_for(cls)
        cv2.fillPoly(overlay, [pts], col)
        cv2.polylines(out, [pts], True, col, 2)
        cv2.putText(out, names[cls], tuple(pts[0]), cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, col, 2, cv2.LINE_AA)
    cv2.addWeighted(overlay, 0.35, out, 0.65, 0, dst=out)
    return out

def banner(img, text):
    out = img.copy()
    cv2.rectangle(out, (0, 0), (out.shape[1], 36), (0, 0, 0), -1)
    cv2.putText(out, text, (10, 26), cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (255, 255, 255), 2, cv2.LINE_AA)
    return out

def side_by_side(gt_img, pred_img, title):
    th = 480
    g = cv2.resize(gt_img, (int(gt_img.shape[1] * th / gt_img.shape[0]), th))
    p = cv2.resize(pred_img, (int(pred_img.shape[1] * th / pred_img.shape[0]), th))
    g = banner(g, 'GT')
    p = banner(p, 'Pred')
    sep = np.full((th, 4, 3), 255, dtype=np.uint8)
    side = np.hstack([g, sep, p])
    return banner(side, title)

bins = {cid: {'good': [], 'missed': [], 'false': []} for cid in range(len(names))}
for img_path, pred in zip(test_imgs, predictions):
    lbl = DATASET / 'test' / 'labels' / (img_path.stem + '.txt')
    gt_classes = set()
    if lbl.exists():
        for line in lbl.read_text().splitlines():
            line = line.strip()
            if line:
                gt_classes.add(int(line.split()[0]))

    if pred.boxes is None or len(pred.boxes) == 0:
        pred_max_conf = {}
    else:
        cls_arr = pred.boxes.cls.cpu().numpy().astype(int)
        conf_arr = pred.boxes.conf.cpu().numpy()
        pred_max_conf = {}
        for c, f in zip(cls_arr, conf_arr):
            pred_max_conf[int(c)] = max(pred_max_conf.get(int(c), 0.0), float(f))
    pred_classes = set(pred_max_conf)

    for cid in range(len(names)):
        in_gt = cid in gt_classes
        in_pred = cid in pred_classes
        score = pred_max_conf.get(cid, 0.0)
        if in_gt and in_pred:
            bins[cid]['good'].append((score, img_path, pred))
        elif in_gt and not in_pred:
            bins[cid]['missed'].append((0.0, img_path, pred))
        elif not in_gt and in_pred:
            bins[cid]['false'].append((score, img_path, pred))

summary = []
for cid, name in enumerate(names):
    safe = name.replace(' ', '_').replace('/', '_')
    counts = {}
    for kind in ('good', 'missed', 'false'):
        items = bins[cid][kind]
        items.sort(key=lambda x: x[0], reverse=True)
        counts[kind] = len(items)
        for rank, (score, img_path, pred) in enumerate(items[:N_PER_BIN], 1):
            img = cv2.imread(str(img_path))
            gt_vis = draw_gt(img, DATASET / 'test' / 'labels' / (img_path.stem + '.txt'))
            pred_vis = pred.plot()
            title = f'[{kind.upper()}] cls={name} conf={score:.2f} file={img_path.stem}'
            out = side_by_side(gt_vis, pred_vis, title)
            cv2.imwrite(str(QUAL_DIR / f'{safe}__{kind}__{rank:02d}__{img_path.stem}.jpg'), out)
    summary.append((name, counts['good'], counts['missed'], counts['false']))

print()
print(f"{'class':<20} {'good':>5} {'missed':>7} {'false':>6}")
print('-' * 42)
for name, g, m, f in summary:
    print(f'{name:<20} {g:>5} {m:>7} {f:>6}')
print()
print('Saved qualitative samples to', QUAL_DIR.resolve())

## 7. Export to dual-output ONNX (browser-ready)

`dynamic=False` for WebGPU fixed shapes; `nms=False` because the TS client runs NMS.

In [ ]:
import hashlib, shutil, json

MODEL_DIR = Path('model')
if MODEL_DIR.exists():
    shutil.rmtree(MODEL_DIR)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy(onnx_path, MODEL_DIR / 'yolo26m-seg-fp16.onnx')
shutil.copy(BEST, MODEL_DIR / 'best.pt')

# Training plots / curves from TRAIN_DIR.
for f in ('results.png', 'results.csv', 'confusion_matrix.png',
          'confusion_matrix_normalized.png',
          'BoxPR_curve.png', 'BoxP_curve.png', 'BoxR_curve.png', 'BoxF1_curve.png',
          'MaskPR_curve.png', 'MaskP_curve.png', 'MaskR_curve.png', 'MaskF1_curve.png'):
    src = TRAIN_DIR / f
    if src.exists():
        shutil.copy(src, MODEL_DIR / f)

# Per-class metrics CSV (lives in TEST_DIR).
if PER_CLASS_CSV.exists():
    shutil.copy(PER_CLASS_CSV, MODEL_DIR / 'per_class_metrics.csv')

# Qualitative samples (lives in TEST_DIR).
if QUAL_DIR.exists():
    shutil.copytree(QUAL_DIR, MODEL_DIR / 'qualitative')

# Test-eval plots from TEST_DIR.
for f in ('confusion_matrix.png', 'confusion_matrix_normalized.png',
          'BoxPR_curve.png', 'MaskPR_curve.png'):
    src = TEST_DIR / f
    if src.exists():
        shutil.copy(src, MODEL_DIR / f'test_{f}')

h = hashlib.sha256()
with open(MODEL_DIR / 'yolo26m-seg-fp16.onnx', 'rb') as fh:
    for chunk in iter(lambda: fh.read(1 << 20), b''):
        h.update(chunk)
sha = h.hexdigest()

with open(MODEL_DIR / 'classes.json', 'w') as fh:
    json.dump({i: n for i, n in enumerate(names)}, fh, indent=2)

with open(MODEL_DIR / 'metadata.yaml', 'w') as fh:
    fh.write(f'''model:
  name: yolo26m-seg-pipevision-fp16
  format: onnx
  input_shape: [1, 3, 640, 640]
  precision: fp16
  opset: 17
  outputs:
    - name: detections
      shape: [1, {expected_ch}, 8400]
    - name: prototypes
      shape: [1, 32, 160, 160]
  sha256: "{sha}"
mask:
  channels: 32
  resolution: 160
inference:
  conf_threshold: 0.25
  iou_threshold: 0.45
  max_detections: 100
metrics:
  test:
    mAP_box_0_5: {test_metrics.box.map50:.4f}
    mAP_box_0_5_0_95: {test_metrics.box.map:.4f}
    mAP_mask_0_5: {test_metrics.seg.map50:.4f}
    mAP_mask_0_5_0_95: {test_metrics.seg.map:.4f}
''')

print('SHA-256 :', sha)
print('staged  :', MODEL_DIR.resolve())
print()
print('top-level files:')
for p in sorted(MODEL_DIR.iterdir()):
    if p.is_file():
        print(f'  {p.stat().st_size / 1e6:8.2f} MB  {p.name}')
    else:
        n_sub = sum(1 for _ in p.iterdir())
        print(f'           dir  {p.name}/  ({n_sub} files)')
print()
print('Download: right-click "model" in the SageMaker file browser → Download.')

## 8. Stage artifacts locally (for download)

Copies the final files into `model/` next to this notebook (overwrites previous runs):
- `best.pt`, `yolo26m-seg-fp16.onnx`, `classes.json`, `metadata.yaml`
- Training plots (`results.png`, confusion matrix, PR curves)
- `per_class_metrics.csv`
- `qualitative/` (good/missed/false samples per class)

Right-click `model/` in the SageMaker file browser → Download.
Then run `sagemaker_hf_upload.ipynb` if you want to publish to HuggingFace.

In [ ]:
import hashlib, shutil, json

MODEL_DIR = Path('model')
if MODEL_DIR.exists():
    shutil.rmtree(MODEL_DIR)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy(onnx_path, MODEL_DIR / 'yolo26m-seg-fp16.onnx')
shutil.copy(BEST, MODEL_DIR / 'best.pt')

# Training plots / curves.
for f in ('results.png', 'results.csv', 'confusion_matrix.png',
          'confusion_matrix_normalized.png',
          'BoxPR_curve.png', 'BoxP_curve.png', 'BoxR_curve.png', 'BoxF1_curve.png',
          'MaskPR_curve.png', 'MaskP_curve.png', 'MaskR_curve.png', 'MaskF1_curve.png'):
    src = Path('runs/segment/tier2_seg_sagemaker') / f
    if src.exists():
        shutil.copy(src, MODEL_DIR / f)

# Per-class metrics CSV.
if PER_CLASS_CSV.exists():
    shutil.copy(PER_CLASS_CSV, MODEL_DIR / 'per_class_metrics.csv')

# Qualitative samples.
if QUAL_DIR.exists():
    shutil.copytree(QUAL_DIR, MODEL_DIR / 'qualitative')

# Test-eval plots (confusion matrix from test split).
for f in ('confusion_matrix.png', 'confusion_matrix_normalized.png',
          'BoxPR_curve.png', 'MaskPR_curve.png'):
    src = Path('runs/segment/tier2_seg_test') / f
    if src.exists():
        shutil.copy(src, MODEL_DIR / f'test_{f}')

h = hashlib.sha256()
with open(MODEL_DIR / 'yolo26m-seg-fp16.onnx', 'rb') as fh:
    for chunk in iter(lambda: fh.read(1 << 20), b''):
        h.update(chunk)
sha = h.hexdigest()

with open(MODEL_DIR / 'classes.json', 'w') as fh:
    json.dump({i: n for i, n in enumerate(names)}, fh, indent=2)

with open(MODEL_DIR / 'metadata.yaml', 'w') as fh:
    fh.write(f'''model:
  name: yolo26m-seg-pipevision-fp16
  format: onnx
  input_shape: [1, 3, 640, 640]
  precision: fp16
  opset: 17
  outputs:
    - name: detections
      shape: [1, {expected_ch}, 8400]
    - name: prototypes
      shape: [1, 32, 160, 160]
  sha256: "{sha}"
mask:
  channels: 32
  resolution: 160
inference:
  conf_threshold: 0.25
  iou_threshold: 0.45
  max_detections: 100
metrics:
  test:
    mAP_box_0_5: {test_metrics.box.map50:.4f}
    mAP_box_0_5_0_95: {test_metrics.box.map:.4f}
    mAP_mask_0_5: {test_metrics.seg.map50:.4f}
    mAP_mask_0_5_0_95: {test_metrics.seg.map:.4f}
''')

print('SHA-256 :', sha)
print('staged  :', MODEL_DIR.resolve())
print()
print('top-level files:')
for p in sorted(MODEL_DIR.iterdir()):
    if p.is_file():
        print(f'  {p.stat().st_size / 1e6:8.2f} MB  {p.name}')
    else:
        n_sub = sum(1 for _ in p.iterdir())
        print(f'           dir  {p.name}/  ({n_sub} files)')
print()
print('Download: right-click "model" in the SageMaker file browser → Download.')

## 9. Shut down

Uncomment to stop the instance. Better to stop manually from the console after downloading.

In [ ]:
# !sudo shutdown -h now